# 16 - Reading the cold constants

**Purpose.** To explain what notebook `15` established about `g`, `R` and the pedestal at -20 C,
and what a reader should now believe about each. `15` is the notebook that talked to the camera
and made these numbers, and is written for someone *checking* the work. This one is written for
someone *deciding what to do next* - specifically, whether session 06's sky frames can be turned
into electrons, and with which numbers.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `cold_constants.json`, `cold_rungs.csv`, `cold_bias.csv`, and the predecessors they
are judged against: `ptc_constants.json` and `ptc_gain.csv` from session 02, `bias_constants.json`
from session 01. If any of it disagreed with `results/`, `results/` would be right and this
notebook would be the bug.

**It assumes `00_statistics.ipynb`** for why a plane mean over a quarter of a million pixels
resolves a hundredth of a count, and why a spread across repeats is the yardstick a difference
has to beat. It assumes `06_ptc_read.ipynb` for what a photon transfer curve measures and why the
pair difference is the thing that makes it blind to fixed pattern.

**The headline is computed below, not typed here.** This notebook was written before the session
ran, so section 1 asks the four questions and the cell under it answers them from the published
file. That is deliberate: a headline typed in advance is a prediction wearing a conclusion's
clothes, and this repo has a word for those.

The four questions, in the order that decides what happens to session 06:

1. **Did the bench hold still?** The two -10 C arms bracket the -20 C one. If they disagree, this
   session measured drift and nothing below it means anything.
2. **Does this bench still reproduce session 02?** Five weeks and one panel reconfiguration
   later, at the two gains that matter.
3. **Did the constants actually move with temperature?** And by more than the bench's own
   wobble - a change smaller than `arm_disagreement` is not a change this session can see.
4. **What does it cost session 06 to have been shot at the wrong setpoint?** Priced in the one
   place it lands: `F_sky`.

In [ ]:
import json
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})

RESULTS = pathlib.Path("..") / "results"
read = lambda n: json.loads((RESULTS / n).read_text())

K7 = read("cold_constants.json")          # notebook 15, this session
K2 = read("ptc_constants.json")           # session 02, at -10 C
K1 = read("bias_constants.json")          # session 01

rungs = pd.read_csv(RESULTS / "cold_rungs.csv")
bias = pd.read_csv(RESULTS / "cold_bias.csv")

num = lambda d: {int(k): v for k, v in d.items()}
GAINS = sorted(num(K7["system_gain_cold"]["value"]))
COLD_C, WARM_C = -20.0, -10.0
PLANES = ["R", "G1", "G2", "B"]

G_COLD = num(K7["system_gain_cold"]["value"])
G_WARM = {g: K7["vs_session02"]["value"][str(g)]["g_this"] for g in GAINS}
G_S02 = num(K2["system_gain"]["value"])
R_COLD = num(K7["read_noise_cold"]["value"])
PED_COLD = num(K7["pedestal_cold"]["value"])
COEF = K7["temperature_coefficient"]["value"]
RESOLVED = K7["temperature_coefficient_resolved"]["value"]
WOBBLE = K7["arm_disagreement"]["value"]

print("notebook 15 published %d constants on %s, from %d frames"
      % (len(K7), K7["system_gain_cold"]["measured_on"],
         K7["system_gain_cold"]["source_frames"]))
print("gains: %s    setpoints: %s" % (GAINS, sorted(rungs.setpoint_c.unique())))
print("rung table: %d rows over %d arms and %d planes"
      % (len(rungs), rungs.arm.nunique(), rungs.plane.nunique()))
print("bias table: %d rows, %d frames per block"
      % (len(bias), int(bias.n_frames.iloc[0])))
print()
print("cooler duty at the end of a -20 C block: %s%%  (gate 2's bar was 90%%)"
      % K7["cooler_duty_at_cold"]["value"])

## 1. The four questions, answered from the file

Read this cell and you have the session. Everything below it is the working.

In [ ]:
warm_repeat = {q: WOBBLE[q] for q in WOBBLE}
q1 = {g: WOBBLE["g"][str(g)] / G_WARM[g] * 100 for g in GAINS}
q2 = {g: (G_WARM[g] / G_S02[g] - 1) * 100 for g in GAINS}

print("1. DID THE BENCH HOLD STILL?")
for g in GAINS:
    print(f"   gain {g:3d}: the two -10 C arms differ by {q1[g]:.2f}% in g")
print("   " + ("steady -- the -20 C arm between them is clean"
               if max(q1.values()) < 0.5 else
               "NOT steady.  Read every coefficient below as a description of the bench"))

print("\n2. DOES THIS BENCH STILL REPRODUCE SESSION 02?")
for g in GAINS:
    print(f"   gain {g:3d}: {G_WARM[g]:.4f} here vs {G_S02[g]:.4f} in session 02  "
          f"({q2[g]:+.2f}%)")
print("   " + ("reproduced -- two sittings five weeks apart agree, which is stronger "
               "evidence than either alone"
               if max(abs(v) for v in q2.values()) < 1.0 else
               "NOT reproduced.  Something changed between the sessions, and the -20 C "
               "numbers are not publishable until it is found.  Do not average the two"))

print("\n3. DID THE CONSTANTS MOVE WITH TEMPERATURE?")
for q in COEF:
    for g in GAINS:
        d, res = COEF[q][str(g)], RESOLVED[q][str(g)]
        print(f"   {q:>9} gain {g:3d}: {d:+.4f}  "
              f"({'resolved' if res else 'inside the bench wobble -- not resolved'})")

print("\n4. WHAT DID THE WRONG SETPOINT COST SESSION 06?")
print("   priced in section 5, where it lands: F_sky.")

## 2. Why the arms are bracketed instead of interleaved, and what that buys

Every other multi-arm session in this repo interleaves, because session 04 learned the hard way
what happens when two arms differ in *what* is tested and in *when* they ran: one of them turned
out to have been warming, and nothing in the data could separate the two.

Here interleaving is not available. A TEC needs minutes to cross 10 C, so a frame-by-frame
rotation would spend the session in transit and none of it in band.

**Bracketing is the substitute, and it is a weaker instrument with an honest error bar.** Arms 1
and 3 sit either side of the -20 C leg. Anything that drifted across the evening - the backlight
warming, the room, the camera - lands in the gap between them. So the uncertainty on every
temperature coefficient is **half that gap**, not the scatter inside a single arm, which is
typically ten times smaller and would flatter the result enormously.

The plot below is the whole argument: the two warm arms against the cold one, per gain, with the
warm pair's own spread drawn as the bar that a temperature effect has to clear.

In [ ]:
per = (rungs[rungs.usable] if "usable" in rungs else rungs)
fig, ax = plt.subplots(1, len(GAINS), figsize=(4.6 * len(GAINS), 3.2))
ax = np.atleast_1d(ax)

for k, g in enumerate(GAINS):
    d = bias[bias.gain == g]
    for arm, mark in zip(sorted(d.arm.unique()), "os^"):
        s = d[d.arm == arm]
        ax[k].scatter(range(len(s)), s.pedestal, marker=mark,
                      label=f"{arm} ({s.setpoint_c.iloc[0]:.0f} C)")
    ax[k].set_xticks(range(len(PLANES)))
    ax[k].set_xticklabels(PLANES)
    ax[k].set_title(f"gain {g}: pedestal per plane per arm")
    ax[k].set_ylabel("ADC counts")
    ax[k].legend(fontsize=7)
fig.tight_layout()

print("the same numbers, and the bar each coefficient has to clear:")
comp = pd.DataFrame({
    "coefficient": {q: COEF[q][str(GAINS[0])] for q in COEF},
    "bench_wobble": {q: WOBBLE[q][str(GAINS[0])] for q in COEF},
    "resolved": {q: RESOLVED[q][str(GAINS[0])] for q in COEF},
})
print(f"at gain {GAINS[0]}:")
print(comp.round(5).to_string())

## 3. `g` should not have moved, and here is why that was the prediction

System gain is set by the sense-node capacitance and the ADC reference. Neither has a strong
temperature coefficient over ten degrees, so the honest prediction written into
`protocols/07-cold-constants.md` before the data existed was **`g` barely moves**.

That makes it a good null to have measured rather than assumed. If `g` is flat, the substitution
session 06 would otherwise have made was harmless - and *knowing* it was harmless is a different
thing from hoping so. If it is not flat, every electron figure in session 06 uses the cold value
and the coefficient is published so that the next accidental setpoint costs an hour instead of an
evening.

The curves below are the raw evidence: variance against signal, one line per arm, at each gain.
Same rungs, same panel, same ROI - only the temperature differs.

In [ ]:
fig, ax = plt.subplots(1, len(GAINS), figsize=(4.6 * len(GAINS), 3.4))
ax = np.atleast_1d(ax)

for k, g in enumerate(GAINS):
    d = rungs[(rungs.gain == g) & rungs.usable]
    for arm, mark in zip(sorted(d.arm.unique()), "os^"):
        s = d[d.arm == arm].groupby("rung").agg(
            signal=("signal", "mean"), var_pair=("var_pair", "mean"),
            setpoint_c=("setpoint_c", "first")).sort_values("signal")
        ax[k].plot(s.signal, s.var_pair, mark + "-", ms=3, lw=0.8,
                   label=f"{arm} ({s.setpoint_c.iloc[0]:.0f} C)")
    ax[k].set(xscale="log", yscale="log", xlabel="signal, ADC counts",
              ylabel="pair variance, counts^2", title=f"gain {g}")
    ax[k].legend(fontsize=7)
fig.tight_layout()

print("g per arm and gain, mean over the four CFA planes (e- per ADC count):")
gt = bias[["arm", "gain"]].drop_duplicates()
tbl = pd.DataFrame({"g_cold": pd.Series(G_COLD), "g_warm_mean": pd.Series(G_WARM),
                    "g_session02": pd.Series(G_S02).reindex(GAINS)})
tbl["cold_vs_warm_pct"] = 100 * (tbl.g_cold / tbl.g_warm_mean - 1)
tbl["warm_vs_s02_pct"] = 100 * (tbl.g_warm_mean / tbl.g_session02 - 1)
print(tbl.round(4).to_string())

## 4. Read noise should have fallen, and the pedestal is the one that matters

Two predictions, and they carry very different consequences.

**`R` falling a few percent is ordinary** - read noise has a thermal component and colder is
quieter. It also barely matters to session 06: the model's read term is `R^2/t`, and at 30 s and
120 s subs under a sky of 1.6 e-/px/s the sky shot noise is already an order of magnitude above
it. A few percent on `R` moves no decision.

**The pedestal is the one that can ruin a number.** `F_sky` is defined on the pedestal-subtracted
frame, so the error goes in count for count. At gain 50 the published -10 C pedestal is about 65
counts, and 120 s of L32's green sky is about 35 counts above it - so **a one-count pedestal
error is a 3% error in the sky rate**, and 3% is inside the night-to-night variation L32 itself
reports. That arithmetic is the whole reason this session was worth an evening, and the cell
below runs it on the measured numbers rather than the illustrative ones.

In [ ]:
L32_GREEN_E_PER_S = 1.594        # inherited, not ours yet -- a prediction being priced, not used
SUBS_S = [30.0, 120.0]

rows = []
for g in GAINS:
    dped = COEF["pedestal"][str(g)]
    for t in SUBS_S:
        sky_counts = L32_GREEN_E_PER_S * t / G_COLD[g]
        rows.append({"gain": g, "sub_s": t, "sky_above_pedestal_counts": sky_counts,
                     "pedestal_shift_counts": dped,
                     "F_sky_error_pct": 100 * dped / sky_counts})
price = pd.DataFrame(rows)
print("what the pedestal shift alone would have done to F_sky, had it been ignored:")
print(price.round(4).to_string(index=False))

print("\nread noise, and how little of the sub it accounts for:")
rn = []
for g in GAINS:
    R_e = K7["read_noise_cold_e"]["value"][str(g)]
    for t in SUBS_S:
        sky_e = L32_GREEN_E_PER_S * t
        rn.append({"gain": g, "sub_s": t, "R_e": R_e, "sky_shot_e": np.sqrt(sky_e),
                   "read_share_of_variance": R_e ** 2 / (R_e ** 2 + sky_e)})
print(pd.DataFrame(rn).round(4).to_string(index=False))

## 5. What the wrong setpoint actually cost session 06

The question this whole session exists to answer, stated as arithmetic rather than as a feeling.

Two numbers are compared for each of session 06's four cells: `F_sky` computed the way it would
have been with session 02's -10 C constants, and `F_sky` computed with this session's -20 C ones.
The difference is the error that would have been published, silently, with no way for a later
reader to detect it.

**It is priced on L32's sky rate, which is a prediction and not ours.** The point is the *size*
of the correction, not the sky rate itself - session 06 measures that, and this cell is only
showing what an uncorrected number would have been wrong by.

In [ ]:
CELLS = [(50, 30.0), (50, 120.0), (200, 30.0), (200, 120.0)]

rows = []
for g, t in CELLS:
    sky_counts_cold = L32_GREEN_E_PER_S * t / G_COLD[g]
    # What the wrong path would have done: measure the same raw level, subtract
    # the -10 C pedestal, and scale by the -10 C g.
    raw = PED_COLD[g] + sky_counts_cold
    ped_warm = PED_COLD[g] - COEF["pedestal"][str(g)]
    wrong_e = (raw - ped_warm) * G_S02[g] / t
    right_e = sky_counts_cold * G_COLD[g] / t
    rows.append({"gain": g, "sub_s": t, "F_sky_right": right_e, "F_sky_wrong": wrong_e,
                 "error_pct": 100 * (wrong_e / right_e - 1)})

cost = pd.DataFrame(rows)
print(cost.round(4).to_string(index=False))
worst = float(cost.error_pct.abs().max())
print(f"\nworst error avoided: {worst:.2f}% on F_sky")
print("  " + ("that is inside L32's own night-to-night variation of about 5%, so the "
              "substitution would have been survivable -- and this session is what makes "
              "'survivable' a measurement instead of a hope"
              if worst < 5 else
              "that is larger than L32's night-to-night variation, so the substitution would "
              "have put a real error into a published constant with nothing to reveal it"))

## 6. What the bench itself did, and the one thing it cannot tell us

Three things worth knowing about the evening, none of them the point of the session and all of
them cheap because the frames were being taken anyway.

**`t_sat` per arm** is three readings at one patch colour. Any drift in the backlight across the
evening appears here, and it appears *before* it appears inside a gain - which is what makes it
useful rather than decorative. Note that `t_sat` legitimately moves with temperature too, through
the pedestal that sets the headroom, so a small tilt is not automatically the panel.

**The cooler duty at -20 C** is the number that says whether this rig could work at that setpoint
rather than merely reach it. Gate 2's bar was 90%, and the margin below it is the headroom for the
sensor self-heating that an hour of readout causes.

**The offset state** turned up in some fraction of the bias blocks and was excluded from both the
pedestal and `R` before either was taken. This session does not study it - session 04 did - but it
does have to survive it, and the count is published so the reader can see it was handled rather
than hoped away.

In [ ]:
tsat = pd.DataFrame(K7["t_sat_per_arm"]["value"]).T
tsat.index.name = "arm"
print("t_sat per arm, seconds, at session 05's patch colour:")
print(tsat.round(4).to_string())
spread = 100 * (tsat.max() - tsat.min()) / tsat.mean()
print("spread across arms, %:")
print(spread.round(3).to_string())

print("\noffset state in the bias blocks:")
st = bias.groupby(["arm", "gain"]).agg(
    far_frac=("state_far_frac", "first"), n_near=("n_near", "first"),
    separation=("state_separation", "first"))
print(st.round(4).to_string())
print(f"\nblocks with a frame in a far state: "
      f"{int((st.far_frac > 0).sum())} of {len(st)}")

## 7. What is settled, and what session 06 may now do

The decision this notebook exists to support, spelled out.

In [ ]:
steady = max(WOBBLE["g"][str(g)] / G_WARM[g] for g in GAINS) < 0.005
repeats = max(abs(G_WARM[g] / G_S02[g] - 1) for g in GAINS) < 0.01

print("SETTLED")
print(f"  g at -20 C, gains {GAINS}: " + ", ".join(f"{G_COLD[g]:.4f}" for g in GAINS)
      + " e-/ADC count")
print(f"  R at -20 C, ADC counts:    " + ", ".join(f"{R_COLD[g]:.4f}" for g in GAINS))
print(f"  pedestal at -20 C, counts: " + ", ".join(f"{PED_COLD[g]:.4f}" for g in GAINS))
print()
print("SESSION 06 MAY NOW " + ("PROCEED" if steady and repeats else "NOT PROCEED"))
if steady and repeats:
    print("  The bench held still across the evening and still reproduces session 02, so the")
    print("  -20 C constants are anchored.  17_sky_pair.ipynb reads cold_constants.json for")
    print("  the pedestal and g, and no substitution is being made.")
else:
    print("  " + ("the two warm arms disagree" if not steady else "")
          + ("; " if not steady and not repeats else "")
          + ("this session does not reproduce session 02" if not repeats else ""))
    print("  Reshoot before publishing any electron figure from session 06.")
print()
print("NOT SETTLED, AND DELIBERATELY OUT OF SCOPE")
print("  The shape of the coefficient between -20 and -10 C: two points, so a straight line is")
print("  an assumption and not a measurement.  Nothing in this repo interpolates it, and")
print("  nothing should until a third setpoint exists.")
print("  Whether -20 C is a better setpoint than -10 C.  MISSION fixes -10 C and this session")
print("  characterises an accident; it does not argue for adopting one.")